In [ ]:
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install('pyb2d-jupyterlite-backend==0.4.2')

# Pyodide's b2d package references an optional module that is absent from its
# wheel. Register only the missing symbol before loading the JupyterLite backend.
import types
from pathlib import Path
import b2d

_compat_name = "b2d.testbed.backend.jupyter.async_jupyter_gui"
_compat_path = (
    Path(b2d.__file__).parent
    / "testbed/backend/jupyter/async_jupyter_gui.py"
)
if not _compat_path.exists() and _compat_name not in sys.modules:
    _compat_module = types.ModuleType(_compat_name)
    _compat_module.JupyterAsyncGui = object
    sys.modules[_compat_name] = _compat_module

# The legacy renderer is not safe inside modern Pyodide workers. Keep
# physics code runnable while suppressing unsupported debug callbacks.
import pyb2d_jupyterlite_backend.plot as _backend_plot
from pyb2d_jupyterlite_backend.async_jupyter_gui import JupyterAsyncGui as _BackendGui
from IPython.display import display

_original_render_world = _backend_plot.render_world
_original_animate_world = _backend_plot.animate_world
_original_gui_init = _BackendGui.__init__

def _render_supported_layers(*args, **kwargs):
    kwargs["flags"] = []
    return _original_render_world(*args, **kwargs)

def _plot_supported_layers(*args, **kwargs):
    display(_render_supported_layers(*args, **kwargs))

def _animate_supported_layers(*args, **kwargs):
    kwargs["flags"] = []
    return _original_animate_world(*args, **kwargs)

def _gui_without_shape_batches(self, *args, **kwargs):
    _original_gui_init(self, *args, **kwargs)
    self._debug_draw_flags = []

_backend_plot.render_world = _render_supported_layers
_backend_plot.plot_world = _plot_supported_layers
_backend_plot.animate_world = _animate_supported_layers
_BackendGui.__init__ = _gui_without_shape_batches
b2d.plot.render_world = _render_supported_layers
b2d.plot.plot_world = _plot_supported_layers
b2d.plot.animate_world = _animate_supported_layers


# Initialize each scene without starting the legacy infinite asyncio loop.
import pyb2d_jupyterlite_backend.async_jupyter_gui as _backend_gui

def _finite_start_ui(self):
    self.canvas = _backend_gui.Canvas(
        width=self.resolution[0], height=self.resolution[1]
    )
    self.out = _backend_gui.ipywidgets.Output()
    self._setup_ipywidgets_gui()
    self.make_testbed()
    self._events = []
    self._stop = True
    return None

_BackendGui.start_ui = _finite_start_ui
print("PyB2D compatibility mode: scene initialized; legacy interactive drawing is disabled.")


In [ ]:
from b2d.testbed import TestbedBase
import b2d


class NewtonsCradle(TestbedBase):

    name = "newton's cradle"

    def __init__(self, settings=None):
        super(NewtonsCradle, self).__init__(settings=settings)

        # radius of the circles
        r = 1.0
        # length of the rope
        l = 10.0
        # how many balls
        n = 10

        offset = (l + r, 2 * r)
        dynamic_circles = []
        static_bodies = []
        for i in range(n):
            if i + 1 == n:
                position = (offset[0] + i * 2 * r + l, offset[1] + l)
            else:
                position = (offset[0] + i * 2 * r, offset[1])

            circle = self.world.create_dynamic_body(
                position=position,
                fixtures=b2d.fixture_def(
                    shape=b2d.circle_shape(radius=r * 0.90),
                    density=1.0,
                    restitution=1.0,
                    friction=0.0,
                ),
                linear_damping=0.01,
                angular_damping=1.0,
                fixed_rotation=True,
            )
            dynamic_circles.append(circle)

            static_body = self.world.create_static_body(
                position=(offset[0] + i * 2 * r, offset[1] + l)
            )

            self.world.create_distance_joint(
                static_body,
                circle,
                local_anchor_a=(0, 0),
                local_anchor_b=(0, 0),
                max_length=l,
                stiffness=0,
            )

            static_bodies.append(static_body)

In [ ]:
from pyb2d_jupyterlite_backend.async_jupyter_gui import JupyterAsyncGui
s = JupyterAsyncGui.Settings()
s.resolution = [1000,300]
b2d.testbed.run(NewtonsCradle, backend=JupyterAsyncGui, gui_settings=s);